# AraStudy Exp02 (Colab, HF Direct)
Fast path: download required splits/tokenizers directly from Hugging Face, then train one tokenizer+seed run and save outputs to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf arastudy
!git clone https://github.com/faresrafat3/arastudy
%cd /content/arastudy
!pip install -r requirements.txt -q
!pip install -q huggingface_hub

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

Path('data/splits/phase2b').mkdir(parents=True, exist_ok=True)
Path('results/tokenizers/phase2b').mkdir(parents=True, exist_ok=True)

# Splits
for fn in ['train.txt', 'valid.txt', 'test.txt']:
    src = hf_hub_download(repo_id='faresrafat/arastudy-arabic-wikipedia-cleaned', repo_type='dataset', filename=fn)
    Path(f'data/splits/phase2b/{fn}').write_bytes(Path(src).read_bytes())

# Tokenizers
files = [
    ('faresrafat/AraStudy-BPE16K-29M', 'tokenizer.model', 'bpe_16k.model'),
    ('faresrafat/AraStudy-BPE16K-29M', 'tokenizer.vocab', 'bpe_16k.vocab'),
    ('faresrafat/AraStudy-BPE32K-37M', 'tokenizer.model', 'bpe_32k.model'),
    ('faresrafat/AraStudy-BPE32K-37M', 'tokenizer.vocab', 'bpe_32k.vocab'),
    ('faresrafat/AraStudy-Morph8K-25M', 'tokenizer.model', 'morph_bpe_8k.model'),
    ('faresrafat/AraStudy-Morph8K-25M', 'tokenizer.vocab', 'morph_bpe_8k.vocab'),
]
for repo, src_name, dst_name in files:
    src = hf_hub_download(repo_id=repo, filename=src_name)
    Path(f'results/tokenizers/phase2b/{dst_name}').write_bytes(Path(src).read_bytes())

!ls -lah data/splits/phase2b | head -n 10
!ls -lah results/tokenizers/phase2b | head -n 20
!test -f data/splits/phase2b/valid.txt && echo OK_VALID || echo MISSING_VALID

In [ ]:
TOKENIZER = "bpe_16k"
SEED = 123
run_id = f"exp02_{TOKENIZER}_s{SEED}"
print("RUN_ID =", run_id)

!python -m src.training.train_exp01_full \
  --config configs/experiments/exp02_multiseed.yaml \
  --tokenizer-id {TOKENIZER} \
  --seed {SEED} \
  --run-id {run_id}

In [ ]:
import os
import shutil

src_dir = f"results/logs/exp02/{run_id}"
dst_dir = f"/content/drive/MyDrive/arastudy_results/exp02/{run_id}"
os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
print(f"✅ Saved to {dst_dir}")

In [ ]:
!ls -lah /content/drive/MyDrive/arastudy_results/exp02/{run_id}
!test -f /content/drive/MyDrive/arastudy_results/exp02/{run_id}/{TOKENIZER}_summary.json && echo OK_SUMMARY || echo MISSING_SUMMARY